# A3.5 · Validating what comes back

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Pass a fabricated claim through a schema check and then through a ground-truth verifier.

**Why a security engineer needs it.** An unverified claim becomes a shared premise, and a peer message is trusted more than a document it is no safer than. The control it builds is: schema validation plus an independent verifier before any claim propagates.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A tool result re-enters the context as a fact. So does a peer's message. A schema check proves the shape is right and says nothing at all about whether the claim inside it is true.

> **At CyberTravels.** The payments API returns `{"status":"refunded"}`. That is a valid shape and it is not evidence the money moved, and the advisor's hotel recommendation is the same problem in prose. R2.

## 2 · The framework

```
   tool result / peer message
            |
            v
   +------------------+   shape is valid, claim may be false
   |  schema check    |   {"status":"ok","rows":0}  <- conforms perfectly
   +--------+---------+
            v
   +------------------+   the claim, checked against something independent
   |    verifier      |   did the row actually appear in the database?
   +------------------+

   conformance is about the serialiser. accuracy is the expensive part.
```

**Mitigates: T5 Cascading Hallucination · T12 Communication Poisoning · T7 Misaligned Behaviour.**

Everything that comes back into the context is an input: tool results, peer
messages, retrieved documents, a sub-agent's summary. A1.10 and A1.12 both
happened because those inputs were trusted in proportion to how internal they
looked rather than to how checked they were.

Two different checks, and conflating them is the mistake:

**Schema validation** asks *is this the right shape*. Cheap, mechanical, catches
malformed input and injection through a field that was supposed to be an
integer. It is necessary and it proves nothing about truth — a perfectly-formed
JSON object can assert anything.

**Verification** asks *is this claim true, according to something that is true
independently of the agent*. A test that passes. A query whose result you can
re-run. A file that exists. A signature that checks.

The rule that follows: **a claim may not propagate past the hop that produced
it without a verification result attached.** Not "was it plausible" — was it
checked, by what, and what did that return.

That single field is what stops A1.12's cascade, because confidence can no
longer rise as evidence disappears: the evidence field travels with the claim,
and an empty one is visible at every hop.

It is also the answer to A1.16, which is why "ask the model whether it
succeeded" is not a verifier — it is the same component grading its own work.

> **What this control closes.**
>
> Stops a claim propagating without evidence attached. Schema validity is not truth: a well-formed object can assert anything.

## 3 · The check, as a skill

`{"status":"refunded"}` is well-formed and may be false. The skill checks four returns twice — schema, then an independent oracle — and confirms that a claim with no oracle stops as `unverifiable` rather than quietly becoming true.

In [ ]:
# skills/runtime/tool-return-validation-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: tool-return-validation-check
description: >-
  Check what a tool's return value is validated against before the agent acts on
  it — schema, then an independent oracle — and confirm that an unverifiable
  claim stops rather than propagating. Use when a tool's output becomes an
  agent's belief, or when reviewing multi-hop reasoning.
allowed-tools: Read, Grep, Glob
---

# Well-formed is not true

A tool return is untrusted input that arrives wearing the tool's authority. Two
checks are needed and they catch different things: a **schema** catches
malformed, and an **oracle** catches confidently wrong. Systems usually have the
first and treat it as if it were the second.

## When to use this

Any agent that acts on what a tool told it, and any pipeline where one step's
output is the next step's premise.

## Procedure

**1 — Define the schema per tool return.** Types and required fields. This is
the cheap check and it should be automatic; a return that fails it never
reaches the model.

**2 — Identify the oracle for each claim type.** Something independent that can
say true or false: a CVE database, a build, a test run, a second source. Not
another model — a model checking a model measures agreement, not truth.

**3 — Run the four cases.** Schema-perfect and true; schema-perfect and false;
schema-perfect with **no oracle available**; malformed. All four have to be
distinguishable in the output.

**4 — Make "unverifiable" a terminal state.** The third case is the one that
matters. A claim with no oracle must stop as `unverifiable` rather than
defaulting to true — silent promotion is how a hedge becomes a fact three hops
later.

**5 — Confirm only verified claims propagate.** Follow each case for several
hops and record which survive. The unverified ones surviving is the finding.

## Output contract

```json
{
  "tools": [{"name": "str", "schema": true, "oracle": "str|null"}],
  "cases": [{"claim": "str", "schema_ok": true, "oracle_verdict": "true|false|unavailable",
             "state": "verified|refuted|unverifiable|malformed", "propagated": false}],
  "unverifiable_is_terminal": true
}
```

## Failure modes

- **Treating schema conformance as verification.** It is a statement about the
  serialiser.
- **Using a model as the oracle.** Two models agreeing is not evidence.
- **Defaulting unverifiable to true** because the pipeline needs a value. That
  default is the defect.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/runtime/tool-return-validation-check/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/runtime/tool-return-validation-check/scripts/tool_return_validation_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check four tool returns against a schema and against an oracle, and propagate only what verified.

This is the executable half of the `tool-return-validation-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SCHEMA = {"claim": str, "confidence": float, "verified_by": (str, type(None))}

def schema_ok(msg):
    return all(k in msg and isinstance(msg[k], t) for k, t in SCHEMA.items())

GROUND_TRUTH = {"libfoo has no known CVEs": False,      # it has one
                "test_login passes": True}

def verify(claim):
    """Independent: reads ground truth, not the sender's opinion."""
    if claim not in GROUND_TRUTH:
        return None, "no oracle for this claim"
    return GROUND_TRUTH[claim], "checked against the advisory database"

def propagate(msg, hops=3):
    """A claim may not travel without a verification result attached."""
    if not schema_ok(msg):
        return {"stopped": "malformed"}
    result, how = verify(msg["claim"])
    if result is None:
        return {"stopped": "unverifiable", "claim": msg["claim"], "why": how}
    if result is False:
        return {"stopped": "refuted", "claim": msg["claim"], "by": how}
    return {"propagated": msg["claim"], "verified_by": how, "hops": hops}

MESSAGES = [
 {"claim": "libfoo has no known CVEs", "confidence": 0.9, "verified_by": None},
 {"claim": "test_login passes",        "confidence": 0.5, "verified_by": None},
 {"claim": "the refund was approved",  "confidence": 0.99, "verified_by": None},
 {"claim": "libfoo is fine",           "confidence": "high", "verified_by": None},
]
for m in MESSAGES:
    print(f"   schema_ok={str(schema_ok(m)):5s} -> {propagate(m)}")

print()
print("The first message is schema-perfect and confident and false. Schema")
print("validation passed it; the oracle refuted it.")
print()
print("The third is unverifiable - no oracle exists. That is a legitimate")
print("outcome and it must not silently become 'true'. It stops here with a")
print("reason, which is what A1.12's cascade never had.")
assert propagate(MESSAGES[0])["stopped"] == "refuted"
assert propagate(MESSAGES[2])["stopped"] == "unverifiable"
assert "propagated" in propagate(MESSAGES[1])

## What you just proved

Four messages are checked twice. A schema-perfect, high-confidence claim is refuted by the oracle; a claim with no oracle stops with `unverifiable` rather than silently becoming true; a malformed message is caught by the schema; and only the verified claim propagates.

## Your turn

Find one place a sub-agent's output becomes another agent's input and ask what oracle checks it. If the answer is the model's own confidence, that is the component grading its own work.

---

**Next → [A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*